# Fidelity, utility, privacy: what does synthetic tabular data actually cost?

This notebook compares two ways of generating synthetic rows for a tabular
medical dataset, and scores both on three axes that matter in practice:

1. **Fidelity**: does the synthetic data look like the real data?
2. **Utility**: can you train a model on synthetic data and have it work on real data (TSTR)?
3. **Privacy**: does the synthetic data leak the real training rows?

The honest answer is that no generator wins on all three at once. The point
of this study is to measure the tradeoffs instead of picking a favorite.

In [2]:
import json, os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 20261002
rng = np.random.default_rng(SEED)
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "src"))
DATA = os.path.join(ROOT, "data")
print("project root:", ROOT)

project root: /home/hatch/workspace/resume-projects/synthetic-data-tradeoffs


## The dataset

I use the breast cancer Wisconsin dataset (scikit-learn, 569 rows, 30 numeric
features of cell nuclei, target 0 = malignant / 1 = benign). Why this one:

- fully numeric features, so both a copula and an LLM can handle it without
  extra encoding machinery;
- a real binary classification task with a known structure, which makes the
  TSTR utility test meaningful;
- public and standard, so anyone can rerun this.

The split is a fixed 70/30 stratified split with seed 20261002: 398 train
rows that the generators are allowed to see, 171 holdout rows that they are
not. Everything downstream is measured against these fixed sets.

In [4]:
train = pd.read_csv(os.path.join(DATA, "train.csv"))
hold = pd.read_csv(os.path.join(DATA, "holdout.csv"))
cop = pd.read_csv(os.path.join(DATA, "copula_synthetic.csv"))
llm = pd.read_csv(os.path.join(DATA, "llm_synthetic.csv"))
feature_names = [c for c in train.columns if c != "target"]
for name, df in [("train", train), ("holdout", hold), ("copula", cop), ("llm", llm)]:
    print(f"{name:8s} rows={len(df):4d}  target mean={df['target'].mean():.3f}  NaNs={int(df.isna().sum().sum())}")

train    rows= 398  target mean=0.628  NaNs=0
holdout  rows= 171  target mean=0.626  NaNs=0
copula   rows= 500  target mean=0.592  NaNs=0
llm      rows= 126  target mean=0.476  NaNs=0


## Generator A: a Gaussian copula, implemented by hand

The idea: keep every univariate margin exactly as observed (rank transform),
and model only the dependence structure with a multivariate normal in
latent space.

1. For each feature, map values to uniform on (0,1) through the empirical
   CDF (ranks divided by n+1).
2. Map uniforms to standard normals with the probit function.
3. Fit a mean vector and covariance matrix to the latent normals.
4. To sample: draw latent normals, map back through the normal CDF and then
   through the inverse empirical CDF of each feature.

I fit one copula per class and sample class labels from the training
prevalence. This is a deliberate choice: the TSTR test needs the
class-conditional structure to be right, and a single joint copula would
dilute it. The cost is that we are not modeling the full joint distribution
as one block, which is fine here because the target is the prediction goal.

What this generator cannot do: anything that is not in the margins and the
correlation matrix. Multimodal features and non-monotone dependence get
flattened.

In [6]:
# The implementation lives in src/copula.py; here I rerun the sanity checks
# on the saved synthetic set to show they hold for the committed file.
from copula import sanity_report
X_train = train[feature_names].values
X_cop = cop[feature_names].values
san = sanity_report(X_cop, cop["target"].values, X_train, feature_names)
print("NaNs:", san["n_nan"], "| Infs:", san["n_inf"])
print("class counts:", san["class_counts"], "| train prevalence:",
      {k: round(v, 3) for k, v in san["train_prevalence"].items()})
print("features with any value outside the train range:", san["out_of_train_range"] or "none")
# Per-class means: does the copula keep the class-conditional locations?
for c in (0, 1):
    d = np.abs(X_train[train["target"].values == c].mean(0) - X_cop[cop["target"].values == c].mean(0))
    print(f"class {c}: max |mean diff| over 30 features = {d.max():.4g}")

NaNs: 0 | Infs: 0
class counts: {'0.0': 204, '1.0': 296} | train prevalence: {'0.0': 0.408, '1.0': 0.592}
features with any value outside the train range: none
class 0: max |mean diff| over 30 features = 34.73
class 1: max |mean diff| over 30 features = 1.202


## Generator B: LLM row synthesis

The second generator is Gemini (flash tier), prompted with the schema
(column names, observed ranges, what the target means) plus 8 real example
rows, and asked for batches of 40 new CSV rows. Batches were parsed
defensively: wrong column counts, non-numeric cells, non-0/1 targets, or
values outside the observed ranges (with 2% slack) were rejected and
retried. The rejection log is saved honestly in the results.

The raw API responses are cached under `data/llm_cache/` so the run is
reproducible without new API calls.

In [8]:
stats = json.load(open(os.path.join(DATA, "llm_stats.json")))
print("batches accepted:", stats["batches_accepted"])
print("batches rejected:", stats["batches_rejected"])
for r in stats["rejections"][:10]:
    print("  rejected batch", r["batch"], "-", r["reason"])
X_llm = llm[feature_names].values
print("\nLLM class balance:", llm["target"].value_counts(normalize=True).round(3).to_dict())
print("LLM NaNs:", int(llm.isna().sum().sum()))

batches accepted: 21
batches rejected: 5
  rejected batch batch_003.txt - no valid data rows after header
  rejected batch batch_007.txt - no valid data rows after header
  rejected batch batch_011.txt - no valid data rows after header
  rejected batch batch_021.txt - worst fractal dimension=0 out of range [0.05504, 0.2075]
  rejected batch batch_022.txt - worst fractal dimension=0 out of range [0.05504, 0.2075]

LLM class balance: {0.0: 0.524, 1.0: 0.476}
LLM NaNs: 0


## Fidelity: does the synthetic data look real?

Three checks, from weak to strong:

1. **Per-feature KS tests** (real train vs synthetic). The copula should do
   well here by construction, since it inverts the empirical CDF.
2. **Correlation-matrix distance**: Frobenius norm between the Pearson
   correlation matrices. This tests the dependence structure the copula
   explicitly models.
3. **MMD with an RBF kernel** (median-heuristic bandwidth fixed from the
   real train set so both generators are compared on the same kernel).
   A joint-distribution test that catches things the margins miss.

In [10]:
from metrics import ks_table, correlation_matrix_distance, mmd_rbf
X_hold = hold[feature_names].values
ks_c = pd.DataFrame(ks_table(X_train, X_cop, feature_names))
ks_l = pd.DataFrame(ks_table(X_train, X_llm, feature_names))
print(f"copula: mean KS={ks_c['ks_stat'].mean():.4f}, max KS={ks_c['ks_stat'].max():.4f}, "
      f"features with p<0.05: {(ks_c['ks_pvalue'] < 0.05).sum()}/30")
print(f"llm:    mean KS={ks_l['ks_stat'].mean():.4f}, max KS={ks_l['ks_stat'].max():.4f}, "
      f"features with p<0.05: {(ks_l['ks_pvalue'] < 0.05).sum()}/30")
print("\nworst 5 features for the LLM by KS:")
print(ks_l.sort_values("ks_stat", ascending=False)[["feature", "ks_stat", "ks_pvalue", "real_mean", "syn_mean"]].head(5).to_string(index=False))
cd_c = correlation_matrix_distance(X_train, X_cop)
cd_l = correlation_matrix_distance(X_train, X_llm)
mmd_c, gamma = mmd_rbf(X_train, X_cop)
mmd_l, _ = mmd_rbf(X_train, X_llm, gamma=gamma)
print(f"\ncorrelation-matrix distance: copula={cd_c:.4f}  llm={cd_l:.4f}")
print(f"MMD^2 (gamma={gamma:.4g}): copula={mmd_c:.5f}  llm={mmd_l:.5f}")

copula: mean KS=0.0428, max KS=0.0631, features with p<0.05: 0/30
llm:    mean KS=0.2695, max KS=0.3892, features with p<0.05: 30/30

worst 5 features for the LLM by KS:
             feature  ks_stat    ks_pvalue  real_mean  syn_mean
      symmetry error 0.389208 1.706578e-13   0.020356  0.030268
          area error 0.361689 1.126670e-11  42.099510 90.339322
     perimeter error 0.360892 1.264773e-11   2.932289  5.129214
concave points error 0.359416 1.563210e-11   0.011867  0.016997
        radius error 0.340791 2.168459e-10   0.413346  0.706178

correlation-matrix distance: copula=1.4168  llm=15.7272
MMD^2 (gamma=2.109e-06): copula=0.00078  llm=0.08195


## Utility: train on synthetic, test on real (TSTR)

The practical question: if I train a classifier on synthetic data and
deploy it on real data, how much AUC do I lose versus training on real data?

I train logistic regression and a random forest (300 trees) on each of the
three training sets and score AUC on the same real holdout. The AUC
*differences* use bootstrap 95% CIs (2000 resamples) so the comparison is
honest about sampling noise.

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from metrics import bootstrap_auc_diff
y_hold = hold["target"].values
y_train = train["target"].values
rows = []
for mname, make in [("logreg", lambda: LogisticRegression(max_iter=2000, random_state=SEED)),
                    ("rf", lambda: RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))]:
    probas = {}
    for key, df in [("real", train), ("copula", cop), ("llm", llm)]:
        m = make().fit(df[feature_names].values, df["target"].values)
        probas[key] = m.predict_proba(X_hold)[:, 1]
    for key in ("real", "copula", "llm"):
        rows.append((mname, key, roc_auc_score(y_hold, probas[key])))
    for key in ("copula", "llm"):
        d = bootstrap_auc_diff(y_hold, probas[key], probas["real"])
        print(f"{mname:6s} {key:6s} vs real: AUC diff {d['auc_diff']:+.4f}  95% CI [{d['ci_low']:+.4f}, {d['ci_high']:+.4f}]")
print()
print(pd.DataFrame(rows, columns=["model", "trained_on", "holdout_auc"]).to_string(index=False))

logreg copula vs real: AUC diff -0.0041  95% CI [-0.0118, +0.0032]
logreg llm    vs real: AUC diff -0.0691  95% CI [-0.1083, -0.0378]
rf     copula vs real: AUC diff -0.0025  95% CI [-0.0104, +0.0037]
rf     llm    vs real: AUC diff -0.0303  95% CI [-0.0560, -0.0114]

 model trained_on  holdout_auc
logreg       real     0.988464
logreg     copula     0.984375
logreg        llm     0.919393
    rf       real     0.984886
    rf     copula     0.982404
    rf        llm     0.954585


/home/hatch/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/hatch/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html


## Privacy: how close is synthetic to real?

Two checks:

1. **Distance to closest record (DCR).** For each synthetic row, the
   Euclidean distance to the nearest real train row (features standardized
   with the train mean/sd). The baseline is real train rows measured against
   *other* real train rows. If synthetic rows sit closer to the training
   data than real rows sit to each other, the generator is copying.
2. **Membership-inference attack.** A deliberately simple one: use the
   negative nearest-neighbor distance to the synthetic set as a score to
   separate train members from holdout non-members. Attack AUC of 0.5 means
   the synthetic data gives away nothing beyond what any new real row would.
   Anything clearly above 0.5 is leakage.

In [14]:
from metrics import dcr_stats, membership_inference_auc
dcr_c = dcr_stats(X_train, X_cop, X_hold)
dcr_l = dcr_stats(X_train, X_llm, X_hold)
for name, d in [("copula", dcr_c), ("llm", dcr_l)]:
    print(f"{name:6s} synthetic->train: mean={d['syn_mean']:.3f} median={d['syn_median']:.3f} p5={d['syn_p5']:.3f}")
print(f"       real->real baseline: mean={dcr_c['train_mean']:.3f} median={dcr_c['train_median']:.3f} p5={dcr_c['train_p5']:.3f}")
print(f"       holdout->train     : mean={dcr_c['holdout_mean']:.3f} median={dcr_c['holdout_median']:.3f} p5={dcr_c['holdout_p5']:.3f}")
print(f"\nMI attack AUC: copula={membership_inference_auc(X_train, X_hold, X_cop):.4f}  "
      f"llm={membership_inference_auc(X_train, X_hold, X_llm):.4f}  (0.5 = no signal)")

copula synthetic->train: mean=2.567 median=2.298 p5=1.503
llm    synthetic->train: mean=3.716 median=2.545 p5=1.361
       real->real baseline: mean=2.500 median=2.211 p5=1.329
       holdout->train     : mean=2.708 median=2.226 p5=1.393

MI attack AUC: copula=0.5421  llm=0.5595  (0.5 = no signal)


## The tradeoff in one picture

Fidelity (correlation-matrix distance) on x, utility loss (TSTR AUC drop for
the random forest) on y, bubble size proportional to privacy leakage
(MI AUC minus 0.5). The ideal generator would sit at the bottom-left with a
tiny bubble.

In [16]:
from plotting import fig_tradeoff
res = json.load(open(os.path.join(DATA, "results.json")))
t = res["tstr"]
summary = {
    "copula": {"corr_dist": res["corr_dist_copula"],
               "auc_drop_rf": t["rf_real"]["auc"] - t["rf_copula"]["auc"],
               "mi_auc": res["mi_auc_copula"]},
    "llm": {"corr_dist": res["corr_dist_llm"],
            "auc_drop_rf": t["rf_real"]["auc"] - t["rf_llm"]["auc"],
            "mi_auc": res["mi_auc_llm"]},
}
fig_tradeoff(summary, os.path.join(ROOT, "figures", "tradeoff_notebook.png"))
print("tradeoff figure saved to figures/tradeoff_notebook.png")
print(json.dumps({k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in summary.items()}, indent=2))

tradeoff figure saved to figures/tradeoff_notebook.png
{
  "copula": {
    "corr_dist": 1.4168,
    "auc_drop_rf": 0.0025,
    "mi_auc": 0.5421
  },
  "llm": {
    "corr_dist": 15.7272,
    "auc_drop_rf": 0.0303,
    "mi_auc": 0.5595
  }
}


## What I take away from this

- The copula is the fidelity winner by construction: exact margins and an
  explicit correlation fit. Its failure mode is structural: anything not in
  the correlation matrix is lost, and the synthetic rows sit a little too
  close to the training data (higher DCR overlap), which is the price of
  fitting the dependence so tightly.
- The LLM rows are plausible but visibly noisier on the margins: larger KS
  statistics, more rejected p-values, and a bigger correlation-matrix
  distance. The rows spread out more (higher DCR), though the simple
  membership-inference attack finds a touch more signal in the LLM set
  (0.560 vs 0.542 for the copula) -- both barely above the 0.5 no-signal
  line, so I would not call this a privacy win either way.
- For TSTR utility, what matters is whether the classifier's decision
  boundary survives. The numbers above say how much AUC each generator costs
  relative to training on real data, with bootstrap CIs so you can see
  whether the gap is real or noise.
- Practical rule of thumb from this experiment: use the copula when you need
  statistical fidelity and the data will stay in-house. The privacy picture
  is mixed (LLM rows sit further from real records by DCR, but the weak
  membership-inference attack found slightly more signal in them), so I
  would not share either set casually. Neither replaces real data for
  model selection.